# Prediccion de Radiacion Solar con BiLSTM
## Demo de producto — Valle de Aburra, Medellin
### Climate Week 2026 · Emergente

---

Este notebook muestra en tiempo real como funciona el sistema de prediccion de radiacion solar desarrollado por **Emergente** para el Valle de Aburra. El modelo BiLSTM corrige los pronosticos del modelo meteorologico global GFS usando una red neuronal entrenada con mas de 4 anos de datos historicos de la red SIATA.

**Que se demuestra en este notebook:**

| Paso | Descripcion |
|------|-------------|
| 1 | Los datos de pronostico GFS que entran al modelo (ventana de 37 horas) |
| 2 | Como la BiLSTM procesa la secuencia completa con atencion |
| 3 | La prediccion para el 15 de septiembre de 2021 |
| 4 | Comparacion con las mediciones reales de SIATA |

**Fecha objetivo del demo:** 15 de septiembre de 2021  
**Ubicacion:** Valle de Aburra, Medellin (6.25°N, 75.5°W, 1485 m s.n.m.)

In [ ]:
# ── Install required packages (run once in Colab) ─────────────────
import subprocess, sys

_PACKAGES = ['xarray', 'h5netcdf', 'pvlib', 'torch',
             'pandas', 'numpy', 'matplotlib', 'seaborn']
for _pkg in _PACKAGES:
    try:
        __import__(_pkg)
    except ImportError:
        print(f'Installing {_pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])

# ── Standard imports ──────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import torch
import xarray as xr
warnings.filterwarnings('ignore')

# ── Project root detection ────────────────────────────────────────
# Walk up from cwd until we find the _4_LSTM_modules marker folder
_search = os.path.abspath(os.getcwd())
PROJECT_ROOT = None
for _ in range(6):
    if os.path.isdir(os.path.join(_search, '_4_LSTM_modules')):
        PROJECT_ROOT = _search
        break
    _search = os.path.dirname(_search)

if PROJECT_ROOT is None:
    # Manual override — set this path if auto-detection fails
    PROJECT_ROOT = r'C:\Users\isabe\Projects\codigors\carpetasdetrabajo'

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT -> {PROJECT_ROOT}')

# ── Matplotlib style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.color':        '#E8E8E8',
    'grid.linewidth':    0.8,
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'legend.fontsize':   10,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
})

# ── Brand colour palette ──────────────────────────────────────────
COLOR_BILSTM = '#2E86AB'   # blue        — BiLSTM prediction
COLOR_SIATA  = '#1B4332'   # dark green  — real SIATA measurement
COLOR_GFS    = '#E07A5F'   # orange      — GFS raw forecast
COLOR_CS     = '#A8DADC'   # light blue  — clear-sky reference

print('Entorno configurado correctamente')

## Paso 1: Que datos entran al modelo?

El modelo no recibe una sola hora de datos: recibe una **ventana simetrica de 37 horas** centrada en la hora que se quiere predecir. Esto le permite capturar tanto el contexto meteorologico reciente (pasado) como la evolucion esperada del tiempo (futuro segun GFS).

```
<- 18h pasado ----------- hora central ----------- 18h futuro ->

 t-18  t-17  ...  t-2  t-1  [ t=0 ]  t+1  t+2  ...  t+17  t+18
                                 ^
                          HORA A PREDECIR
```

**Para cada una de las 37 horas de la ventana se incluyen 79 variables:**

| Grupo | Variables principales | Fuente |
|---|---|---|
| Radiacion GFS | `dswrf1`, `kc` (CSI), `ks` (KC) | GFS — 4 lanzamientos |
| Nubosidad GFS | `TCDC`, `LCDC`, `MCDC`, `HCDC`, `HGT` | GFS — 4 lanzamientos |
| Meteorologia GFS | `TMP`, `RH`, `CAPE`, `HPBL`, `WIND`, `PWAT`, `DLWRF` | GFS — 4 lanzamientos |
| Solar / meta | `zenith`, `step` | pvlib / GFS |
| Climatologia SIATA | `clim_mean_csi`, `clim_std_csi`, `clim_prob_*`, etc. | SIATA historico |

**Total: 79 variables x 37 pasos = 2.923 valores por prediccion**

A continuacion se muestra la ventana real que el modelo recibio para predecir las 12:00h del 15 de septiembre de 2021.

In [ ]:
# ── File paths (relative to PROJECT_ROOT) ──────────────────────────
_NPZ_CANDIDATES = [
    os.path.join(PROJECT_ROOT, '_4_LSTM_modules', 'Prepared_data',
                 '4launch_multfeat_sym18_clim79_FIXED.npz'),   # spec name
    os.path.join(PROJECT_ROOT, '_4_LSTM_modules', 'Prepared_data',
                 '4launch_multfeat_sym18_clim.npz'),            # actual file
]

# ── Load NPZ ──────────────────────────────────────────────────────
try:
    NPZ_PATH = next(p for p in _NPZ_CANDIDATES if os.path.exists(p))
    data = np.load(NPZ_PATH, allow_pickle=True)
    feature_vars = list(data['feature_vars'])
    print(f'NPZ cargado : {os.path.basename(NPZ_PATH)}')
    print(f'  X_train shape : {data["X_train"].shape}')
    print(f'  Features      : {len(feature_vars)}')
except StopIteration:
    print('ERROR: No se encontro el archivo NPZ.')
    print('  Rutas buscadas:')
    for p in _NPZ_CANDIDATES:
        print(f'    {p}')
    raise

# ── Merge all splits and locate 2021-09-15 12:00 ──────────────────
TARGET_DT = pd.Timestamp('2021-09-15 12:00:00')

X_all = np.concatenate([data['X_train'], data['X_val'], data['X_test']])
y_all = np.concatenate([data['y_train'], data['y_val'], data['y_test']])
t_all = pd.to_datetime(
    np.concatenate([data['t_train'], data['t_val'], data['t_test']])
)

mask_target = (t_all == TARGET_DT)
if not mask_target.any():
    raise ValueError(
        f'No se encontro la secuencia para {TARGET_DT}. '
        'Verificar que el NPZ contiene datos de septiembre 2021.'
    )

idx       = int(np.where(mask_target)[0][0])
X_seq     = X_all[idx]    # shape (37, 79)
n_train   = len(data['X_train'])
n_val     = len(data['X_val'])
split_tag = ('train' if idx < n_train
             else 'val' if idx < n_train + n_val
             else 'test')
print(f'  Secuencia objetivo : {TARGET_DT}  [split: {split_tag}]')
print(f'  Forma de ventana   : {X_seq.shape}')

# ── Select 10 most informative features for the display table ─────
TEN_VARS = [
    'kc_0700', 'ks_0700', 'dswrf1_0700', 'TCDC_ent_0700',
    'LCDC_ent_0700', 'CAPE_surface_0700', 'RH_2m_0700',
    'TMP_surface_0700', 'HPBL_surface_0700', 'zenith',
]
avail_vars  = [v for v in TEN_VARS if v in feature_vars]
col_indices = [feature_vars.index(v) for v in avail_vars]

# Build one timestamp per window step (local Colombia time)
win_start  = TARGET_DT + pd.Timedelta(hours=-18)
timestamps = [win_start + pd.Timedelta(hours=i) for i in range(37)]
paso_label = [f't{i - 18:+d}' if i != 18 else '[t=0]' for i in range(37)]

df_window = pd.DataFrame(
    X_seq[:, col_indices],
    columns=avail_vars,
).round(3)
df_window.insert(0, 'Paso', paso_label)
df_window.index = [ts.strftime('%Y-%m-%d %H:%M') for ts in timestamps]
df_window.index.name = 'Timestamp (local)'

print(f'\nVentana de entrada al modelo: 37 pasos temporales')
print(f'Variables seleccionadas ({len(avail_vars)} de {len(feature_vars)} totales)')

# ── Style the table ───────────────────────────────────────────────
def _style_window_row(row):
    pos = df_window.index.get_loc(row.name)
    if pos < 18:
        return ['background-color: #F5F5F5; color: #555555'] * len(row)
    elif pos == 18:
        return [
            f'background-color: {COLOR_BILSTM}33; '
            'font-weight: bold; border: 2px solid #2E86AB'
        ] * len(row)
    else:
        return ['background-color: #FFFDE7'] * len(row)

styled_window = (
    df_window.style
    .apply(_style_window_row, axis=1)
    .set_caption(
        'Ventana de entrada BiLSTM — 15-sep-2021 12:00h '
        '(fila central en azul = hora a predecir | '
        'gris = pasado | amarillo = futuro)'
    )
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '12px'), ('font-weight', 'bold'), ('color', '#333')]},
        {'selector': 'th',
         'props': [('background-color', COLOR_BILSTM),
                   ('color', 'white'), ('font-size', '11px')]},
        {'selector': 'td',
         'props': [('font-size', '10.5px'), ('padding', '3px 8px')]},
    ])
)
styled_window

## Paso 2: El modelo procesa la secuencia completa

La BiLSTM no analiza las horas de forma independiente: **lee toda la ventana de 37 pasos simultaneamente**, combinando informacion del pasado y del futuro para hacer una prediccion mas precisa del instante central.

**Arquitectura del modelo:**

```
Entrada (37 x 79)
      |
      v
  LayerNorm
      |
   +--+--+
   |     |
   v     v
LSTM->  <-LSTM
(fwd)   (bwd)
   |     |
   +--+--+
      |   (x 3 capas, hidden=96)
      v
 Atencion Bahdanau
 (aprende que horas importan mas)
      |
      v
  Dropout (0.25)
      |
      v
  Capa lineal  -->  Sigmoid  -->  CSI predicho [0,1]
```

**Descalado fisico:**  
`GHI predicho (W/m2) = CSI predicho x GHI cielo despejado (Ineichen)`

El mecanismo de **atencion** aprende a dar mas peso a las horas donde las senales de nubosidad (TCDC, LCDC, CAPE) y radiacion GFS son mas informativas para corregir el pronostico central.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    '_4_LSTM_modules', '_runs', '4launch_multfeat_sym',
    '4launch_Multfeat_sym18_clim79_FIXED_BiLSTM_attn_20260706_102943',
    'best_model.pt',
)
CS_GHI_PATH = os.path.join(
    PROJECT_ROOT,
    '_3_Data_preparation_for_LSTM', 'Preparation_data',
    '_01_CSI_EXT_radiation', 'Ineichen_GHI',
    'CSI_GHI_grid25_avg_with_horizon_and_enhancement_with_bias_correct2.nc',
)

# ── Load model ────────────────────────────────────────────────────
try:
    from _4_LSTM_modules.NN_modules.BiLSTMRegressor import BiLSTMRegressor

    model = BiLSTMRegressor(
        n_feat=79, hidden=96, num_layers=3, dropout=0.25, activation='sigmoid'
    )
    state = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
    model.load_state_dict(state)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Modelo cargado : {n_params:,} parametros')
    print(f'  Archivo       : {os.path.basename(MODEL_PATH)}')
except FileNotFoundError:
    print('ERROR: No se encontro el archivo del modelo:')
    print(f'  {MODEL_PATH}')
    raise
except Exception as e:
    print(f'ERROR al cargar el modelo: {e}')
    raise

# ── Find all sequences for 2021-09-15 (daytime 06-18h) ───────────
mask_day = (
    (t_all >= '2021-09-15 06:00:00') &
    (t_all <= '2021-09-15 18:00:00')
)
order   = np.argsort(t_all[mask_day])
X_day   = X_all[mask_day][order]
y_day   = y_all[mask_day][order]
t_day   = t_all[mask_day][order]
print(f'\nSecuencias encontradas para 2021-09-15: {len(t_day)} de 13 horas diurnas')

# ── Run inference ─────────────────────────────────────────────────
with torch.no_grad():
    X_tensor  = torch.tensor(X_day, dtype=torch.float32)
    csi_pred  = model(X_tensor).numpy()   # shape (13,) — normalised CSI in (0,1)

print(f'Rango CSI predicho : [{csi_pred.min():.3f}, {csi_pred.max():.3f}]')

# ── Physical descaling: GHI = CSI_pred x clear_sky_ghi ───────────
try:
    ds_cs      = xr.open_dataset(CS_GHI_PATH, engine='h5netcdf')
    t_cs_all   = pd.to_datetime(ds_cs['observation_time'].values)
    v_cs_all   = ds_cs['clear_sky_ghi'].values.squeeze()   # (N,) after squeezing spatial dims
    ds_cs.close()

    # Align clear-sky values to the inference timestamps
    cs_series  = pd.Series(v_cs_all, index=t_cs_all)
    cs_values  = cs_series.reindex(t_day).values
    print(f'Clear-sky GHI cargado: rango [{cs_values.min():.0f}, {cs_values.max():.0f}] W/m2')
except FileNotFoundError:
    print('ERROR: No se encontro el archivo de cielo despejado:')
    print(f'  {CS_GHI_PATH}')
    raise

ghi_pred = csi_pred  * cs_values   # W/m2
ghi_true = y_day     * cs_values   # W/m2 (ground truth, used later)

# ── Build results DataFrame ───────────────────────────────────────
df_pred = pd.DataFrame({
    'hora':          [ts.strftime('%H:%M') for ts in t_day],
    'csi_pred':      np.round(csi_pred,  3),
    'ghi_pred':      np.round(ghi_pred,  1),
    'clear_sky_ghi': np.round(cs_values, 1),
    'csi_true':      np.round(y_day,     3),
    'ghi_true':      np.round(ghi_true,  1),
}, index=t_day)
df_pred.index.name = 'timestamp_local'

print(f'\nPredicciones generadas para 2021-09-15')
print(f'Horas con datos: {len(df_pred)} de 13 horas diurnas')

In [ ]:
# ── Styled prediction table ────────────────────────────────────────
from matplotlib.colors import LinearSegmentedColormap

display_df = df_pred[['hora', 'csi_pred', 'ghi_pred', 'clear_sky_ghi']].copy()
display_df.columns = ['Hora', 'CSI predicho', 'GHI predicho (W/m2)', 'Clear-sky (W/m2)']
display_df = display_df.reset_index(drop=True)

# GHI colour gradient: white -> orange
_cm_orange = LinearSegmentedColormap.from_list('wh_orange', ['#FFFFFF', '#E07A5F'], N=256)
_peak_cs   = float(df_pred['clear_sky_ghi'].max()) * 1.05

styled_pred = (
    display_df.style
    .background_gradient(
        cmap=_cm_orange,
        subset=['GHI predicho (W/m2)'],
        vmin=0,
        vmax=_peak_cs,
    )
    .format({'CSI predicho': '{:.3f}',
             'GHI predicho (W/m2)': '{:.0f}',
             'Clear-sky (W/m2)': '{:.0f}'})
    .set_caption('Predicciones BiLSTM — 15 de septiembre 2021')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '13px'), ('font-weight', 'bold'), ('color', '#333')]},
        {'selector': 'th',
         'props': [('background-color', COLOR_BILSTM), ('color', 'white')]},
        {'selector': 'td',
         'props': [('padding', '4px 12px'), ('font-size', '11px')]},
    ])
)

# ── Summary statistics ────────────────────────────────────────────
_mask_pos   = ghi_pred > 0
_avg_ghi    = ghi_pred[_mask_pos].mean()
_peak_idx   = int(np.argmax(ghi_pred))
_peak_hour  = df_pred['hora'].iloc[_peak_idx]
_peak_ghi   = ghi_pred[_peak_idx]

print(f'GHI promedio del dia               : {_avg_ghi:.0f} W/m2')
print(f'Hora de maxima radiacion predicha  : {_peak_hour} — {_peak_ghi:.0f} W/m2')
print()
styled_pred

## Paso 3: Comparacion con la medicion real

El modelo produjo sus predicciones **sin conocer lo que realmente ocurrio** ese dia. Ahora comparamos con las mediciones de la red **SIATA** (Sistema de Alerta Temprana del Valle de Aburra).

**Por que esta comparacion importa:**

| | GFS sin correccion | BiLSTM (nuestro modelo) |
|---|---|---|
| Fuente | Modelo numerico NOAA | Red neuronal + GFS |
| Corrige sesgo sistematico | No | Si |
| Aprende de errores historicos | No | Si |
| Considera contexto temporal | Parcial | Si (ventana de 37h) |

La diferencia entre prediccion y realidad es lo que el entrenamiento busca minimizar.  
En el conjunto de **prueba** (15% de datos que el modelo nunca vio durante el entrenamiento):

| Metrica | GFS crudo | BiLSTM | Mejora |
|---|---|---|---|
| RMSE diurno | 201.9 W/m2 | 128.4 W/m2 | **-36.4%** |
| Skill Score | 0.00 (referencia) | **+0.60** | — |

In [ ]:
# ── File paths ────────────────────────────────────────────────────
SIATA_PATH = os.path.join(
    PROJECT_ROOT,
    '_3_Data_preparation_for_LSTM', 'Preparation_data',
    '_03_Siata_GHI', 'Netcdf_Siata_GHI', 'SIATA_GHI_all.nc',
)
GFS_CSI_PATH = os.path.join(
    PROJECT_ROOT,
    '_3_Data_preparation_for_LSTM', 'Preparation_data',
    '_04_indices', 'clear_sky_indices', 'clearsky_index_GFS_0700.nc',
)

TARGET_DATE = '2021-09-15'

# ── Hourly alignment index (daytime) ─────────────────────────────
HOURS_IDX = pd.date_range('2021-09-15 06:00', '2021-09-15 18:00', freq='1h')

def _to_hourly(timestamps, values, name):
    """Align an arbitrary time series to the daytime hourly grid."""
    s = pd.Series(values, index=pd.to_datetime(timestamps), name=name)
    s = s[~s.index.duplicated(keep='first')]
    return s.reindex(HOURS_IDX)

# ── Load SIATA GHI ────────────────────────────────────────────────
try:
    ds_s    = xr.open_dataset(SIATA_PATH, engine='h5netcdf')
    t_s_all = pd.to_datetime(ds_s['observation_time'].values)
    v_s_all = ds_s['GHI'].values.squeeze()    # variable confirmed: 'GHI'
    ds_s.close()

    _mask_s = (t_s_all >= TARGET_DATE) & (t_s_all < '2021-09-16')
    sr_siata = _to_hourly(t_s_all[_mask_s], v_s_all[_mask_s], 'GHI_SIATA')
    print(f'SIATA cargado  : {_mask_s.sum()} registros para {TARGET_DATE}')
    print(f'  Horas con GHI > 0 : {int((sr_siata > 0).sum())}')
except FileNotFoundError:
    print(f'ERROR: No se encontro SIATA_GHI_all.nc en:\n  {SIATA_PATH}')
    raise

# ── Load GFS raw CSI (launch 07:00) ───────────────────────────────
try:
    ds_g    = xr.open_dataset(GFS_CSI_PATH, engine='h5netcdf')
    gfs_var = list(ds_g.data_vars)[0]          # 'clearsky_index_GFS_0700'
    t_g_all = pd.to_datetime(ds_g['observation_time'].values)
    v_g_all = ds_g[gfs_var].values.squeeze()
    ds_g.close()

    _mask_g  = (t_g_all >= TARGET_DATE) & (t_g_all < '2021-09-16')
    sr_gfs   = _to_hourly(t_g_all[_mask_g], v_g_all[_mask_g], 'CSI_GFS')
    print(f'GFS CSI cargado: {_mask_g.sum()} registros para {TARGET_DATE}  [var: {gfs_var!r}]')
except FileNotFoundError:
    print(f'ERROR: No se encontro clearsky_index_GFS_0700.nc en:\n  {GFS_CSI_PATH}')
    raise

# ── Load clear-sky GHI (reuse path already verified) ─────────────
ds_cs2   = xr.open_dataset(CS_GHI_PATH, engine='h5netcdf')
t_cs2    = pd.to_datetime(ds_cs2['observation_time'].values)
v_cs2    = ds_cs2['clear_sky_ghi'].values.squeeze()
ds_cs2.close()
_mask_c  = (t_cs2 >= TARGET_DATE) & (t_cs2 < '2021-09-16')
sr_cs2   = _to_hourly(t_cs2[_mask_c], v_cs2[_mask_c], 'Clear_sky')

# ── BiLSTM predictions already computed — align to same grid ─────
sr_bilstm = _to_hourly(df_pred.index, df_pred['ghi_pred'].values, 'GHI_BiLSTM')

# ── Build comparison DataFrame ────────────────────────────────────
df_comp = pd.DataFrame({
    'GHI_SIATA':  sr_siata,
    'GHI_BiLSTM': sr_bilstm,
    'GHI_GFS':    sr_gfs * sr_cs2,   # CSI x clear_sky -> W/m2
    'Clear_sky':  sr_cs2,
})
# Keep only hours where clear-sky > 0 and SIATA measured something
df_comp = df_comp[(df_comp['Clear_sky'] > 0) & df_comp['GHI_SIATA'].notna()]

print(f'\nDataFrame de comparacion: {len(df_comp)} horas comunes')
print(df_comp[['GHI_SIATA', 'GHI_BiLSTM', 'GHI_GFS']].round(0).to_string())

In [ ]:
# ── Main visualisation: 4-line comparison figure ──────────────────
fig, ax = plt.subplots(figsize=(14, 7))

# Use integer positions on x-axis to avoid datetime rendering issues
x      = list(range(len(df_comp)))
labels = [ts.strftime('%H:%M') for ts in df_comp.index]

cs_vals     = df_comp['Clear_sky'].values
gfs_vals    = df_comp['GHI_GFS'].values
bilstm_vals = df_comp['GHI_BiLSTM'].values
siata_vals  = df_comp['GHI_SIATA'].values

# Line 1 — Clear-sky reference
ax.plot(x, cs_vals,
        color=COLOR_CS, linewidth=1.8, linestyle='--', alpha=0.9,
        label='Cielo despejado (Ineichen)', zorder=1)

# Line 2 — GFS raw
ax.plot(x, gfs_vals,
        color=COLOR_GFS, linewidth=2.2, alpha=0.85,
        label='GFS sin correccion', zorder=2)

# Line 3 — BiLSTM prediction
ax.plot(x, bilstm_vals,
        color=COLOR_BILSTM, linewidth=2.8,
        label='BiLSTM (nuestro modelo)', zorder=4)

# Line 4 — SIATA real measurement
ax.plot(x, siata_vals,
        color=COLOR_SIATA, linewidth=2.5,
        marker='o', markersize=6, markerfacecolor='white', markeredgewidth=2,
        label='Medicion real SIATA', zorder=5)

# Shaded area: error between BiLSTM and SIATA
ax.fill_between(x, bilstm_vals, siata_vals,
                alpha=0.12, color=COLOR_BILSTM,
                label='_nolegend_', zorder=3)

# Vertical line at noon
_noon_positions = [i for i, ts in enumerate(df_comp.index) if ts.hour == 12]
if _noon_positions:
    ax.axvline(_noon_positions[0], color='#AAAAAA', linestyle='--',
               linewidth=1.2, alpha=0.7, zorder=0)
    ax.text(_noon_positions[0] + 0.08,
            ax.get_ylim()[1] * 0.97 if ax.get_ylim()[1] > 0 else 950,
            '12:00', color='#AAAAAA', fontsize=9, va='top')

# ── Annotations ──────────────────────────────────────────────────
# Find the hour where GFS error is largest (absolute)
_gfs_err    = np.where(~np.isnan(gfs_vals), np.abs(gfs_vals - siata_vals), 0)
_bilstm_err = np.where(~np.isnan(bilstm_vals), np.abs(bilstm_vals - siata_vals), np.inf)
_pos_worst  = int(np.argmax(_gfs_err))
_pos_best   = int(np.argmin(_bilstm_err))

_y_worst = float(gfs_vals[_pos_worst]) if not np.isnan(gfs_vals[_pos_worst]) else 0
_y_best  = float(bilstm_vals[_pos_best]) if not np.isnan(bilstm_vals[_pos_best]) else 0

ax.annotate(
    'GFS sobreestima aqui',
    xy=(_pos_worst, _y_worst),
    xytext=(_pos_worst - 1.8, _y_worst + 130),
    fontsize=9, color=COLOR_GFS, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=COLOR_GFS, lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
              edgecolor=COLOR_GFS, alpha=0.85),
)
ax.annotate(
    'BiLSTM corrige el error',
    xy=(_pos_best, _y_best),
    xytext=(_pos_best + 0.4, _y_best + 160),
    fontsize=9, color=COLOR_BILSTM, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=COLOR_BILSTM, lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
              edgecolor=COLOR_BILSTM, alpha=0.85),
)

# ── Metrics text box (global test-set results) ────────────────────
_metrics_text = (
    'Metricas — conjunto de prueba (test set)\n'
    '─────────────────────────────────────\n'
    'Skill Score BiLSTM :  +0.60\n'
    'RMSE GFS diurno    :  201.9 W/m2\n'
    'RMSE BiLSTM diurno :  128.4 W/m2\n'
    'Mejora             :   36.4 %'
)
ax.text(
    0.015, 0.975, _metrics_text,
    transform=ax.transAxes, va='top', ha='left',
    fontsize=9.5, fontfamily='monospace',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='white',
              edgecolor=COLOR_BILSTM, alpha=0.92, linewidth=1.5),
)

# ── Labels and formatting ─────────────────────────────────────────
ax.set_title(
    'Radiacion Solar — 15 de septiembre 2021\nValle de Aburra, Medellin',
    fontsize=14, fontweight='bold', pad=14,
)
ax.set_xlabel('Hora local (Colombia, UTC-5)', fontsize=11)
ax.set_ylabel('Irradiancia GHI (W/m2)', fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

ax.legend(
    loc='lower center', ncol=4,
    bbox_to_anchor=(0.5, -0.22),
    frameon=True, framealpha=0.95,
    edgecolor='#CCCCCC',
)

fig.tight_layout()

# ── Save figure ───────────────────────────────────────────────────
_fig_path = os.path.join(
    os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir()
    else os.getcwd(),
    'demo_climate_week_sep2021.png',
)
try:
    fig.savefig(_fig_path, dpi=150, bbox_inches='tight')
    print(f'Figura guardada -> {_fig_path}')
except Exception as _e:
    print(f'No se pudo guardar la figura: {_e}')

plt.show()

## Conclusion del demo

### Lo que se demostro en este notebook

1. **Los datos entran como ventanas de 37 horas** — el modelo ve el contexto pasado *y* futuro del pronostico GFS antes de emitir una prediccion para la hora central.

2. **La BiLSTM corrige el sesgo del GFS** — usando un mecanismo de atencion que identifica automaticamente cuales horas de la ventana son mas informativas.

3. **La mejora es medible y consistente** — en el conjunto de prueba independiente (15% de datos que el modelo nunca vio), la reduccion de RMSE diurno es del **36.4%** respecto al GFS crudo.

---

### Las tres capas del sistema Emergente

```
CAPA 1 — GFS (fisica)
  Pronostico numerico global NOAA
  Resolucion: ~25 km, lanzamientos cada 6h
  Limitacion: sesgo alto en regiones nubladas como el Valle de Aburra

CAPA 2 — BiLSTM (correccion ML)  <-- lo que se mostro aqui
  Red neuronal entrenada con 4+ anos de datos SIATA
  Aprende los patrones de error sistematico del GFS para Medellin
  Reduccion RMSE: 36.4% | Skill Score: +0.60

CAPA 3 — GOES-19 (asimilacion satelital)  [PROXIMO]
  Imagenes de nubosidad cada 10 minutos
  Permitira actualizar el pronostico en tiempo casi real
  Objetivo: <5% error en las proximas 2h
```

---

**Repositorio del proyecto:** disponible en el servidor interno Emergente  
**Datos:** Red SIATA — Sistema de Alerta Temprana del Valle de Aburra  
**Modelo GFS:** NOAA Global Forecast System (GFS 0.25°)